# 09 — Plugin Authoring: A Minimal `LanguagePlugin`

Notebook 06 showed the plugin *discovery* mechanism. This notebook goes deeper: we write a minimal **`LanguagePlugin`** for a toy language called **Mini-JS**, register it programmatically (no `pyproject.toml` entry point, no package install), and run it on a three-line JavaScript-like snippet.

The goal is not to replace COGANT's real tree-sitter JS plugin — it's to give you a step-by-step template for the simplest possible plugin so you can adapt it for your own languages.

**Run from the `cogant/` directory:**
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/09_plugin_authoring.ipynb
```

## 1. The `LanguagePlugin` contract

Every language plugin subclasses `cogant.plugins.base.LanguagePlugin` and must implement four abstract methods:

| Method | Purpose |
| --- | --- |
| `parse(source_code)` | Produce an AST-shaped dict from raw source. |
| `extract_symbols(ast)` | Return a list of `{name, kind, ...}` dicts. |
| `extract_types(ast)` | Return a type environment (may be empty). |
| `resolve_imports(ast)` | Return a list of import target strings. |

Plus the `Plugin` lifecycle methods `initialize(config)` and `shutdown()`. That's it — no required configuration, no required base-class wiring.

In [1]:
from __future__ import annotations

import re
from typing import Any

from cogant.plugins.base import LanguagePlugin, PluginMetadata


class MiniJsPlugin(LanguagePlugin):
    """Tiny regex-based plugin for a subset of JavaScript.

    Recognises three constructs:
      - `function NAME(...) { ... }`
      - `const NAME = ...`
      - `import ... from 'TARGET'`
    """

    supported_languages: set[str] = {"minijs"}

    _FN = re.compile(r"function\s+([A-Za-z_][\w]*)\s*\(")
    _CONST = re.compile(r"const\s+([A-Za-z_][\w]*)\s*=")
    _IMPORT = re.compile(r"import\s+.+?\s+from\s+['\"]([^'\"]+)['\"]")

    def __init__(self) -> None:
        super().__init__(
            PluginMetadata(
                name="minijs",
                version="0.1.0",
                author="COGANT tutorials",
                description="Regex-based Mini-JS plugin for notebook 09.",
            )
        )
        self._config: dict[str, Any] = {}

    # --- lifecycle ---------------------------------------------------

    def initialize(self, config: dict[str, Any]) -> None:
        self._config = dict(config)

    def shutdown(self) -> None:
        self._config.clear()

    # --- LanguagePlugin contract ------------------------------------

    def parse(self, source_code: str) -> dict[str, Any]:
        nodes: list[dict[str, Any]] = []
        imports: list[str] = []
        for lineno, line in enumerate(source_code.splitlines(), start=1):
            stripped = line.strip()
            if not stripped or stripped.startswith("//"):
                continue
            if m := self._FN.search(stripped):
                nodes.append({"kind": "function", "name": m.group(1), "lineno": lineno})
            elif m := self._CONST.search(stripped):
                nodes.append({"kind": "const", "name": m.group(1), "lineno": lineno})
            if m := self._IMPORT.search(stripped):
                imports.append(m.group(1))
        return {"language": "minijs", "nodes": nodes, "imports": imports}

    def extract_symbols(self, ast: dict[str, Any]) -> list[dict[str, Any]]:
        return [
            {"name": n["name"], "kind": n["kind"], "line_start": n["lineno"]}
            for n in ast.get("nodes", [])
        ]

    def extract_types(self, ast: dict[str, Any]) -> dict[str, Any]:
        # Mini-JS has no static types — return an empty env.
        return {}

    def resolve_imports(self, ast: dict[str, Any]) -> list[str]:
        return list(ast.get("imports", []))


plugin = MiniJsPlugin()
print(f"Instantiated: {plugin.metadata.name} v{plugin.metadata.version}")

Instantiated: minijs current


## 2. Register the plugin in a `PluginRegistry`

`PluginRegistry` is normally populated from `importlib.metadata` entry points, but the internal caches (`_cache` for `PluginInfo`, `_loaded_objects` for the instances) are directly writable — which is exactly what notebooks and tests need. This is the same pattern used in notebook 06.

In [2]:
from cogant.plugins import PluginInfo, PluginRegistry

registry = PluginRegistry()
discovered = registry.discover()
print(f"Discovered via entry points: {len(discovered)}")

info = PluginInfo(
    name=plugin.metadata.name,
    version=plugin.metadata.version,
    entry_point=f"<in-process>:{type(plugin).__name__}",
    loaded=True,
    error=None,
)
registry._cache[info.name] = info  # noqa: SLF001
registry._loaded_objects[info.name] = plugin  # noqa: SLF001

plugin.initialize({"strict": False})

print("Plugins visible to the registry:", registry.list_plugins())

Discovered via entry points: 0
Plugins visible to the registry: ['minijs']


## 3. Parse a three-line JS snippet

We feed Mini-JS a tiny snippet that exercises every construct it knows about: one `import`, one `const`, and one `function`.

In [3]:
snippet = """
import { Widget } from 'widgets';
const MAX_ITEMS = 10;
function renderList(items) { return items.slice(0, MAX_ITEMS); }
""".strip()

print("Source:")
for i, line in enumerate(snippet.splitlines(), start=1):
    print(f"  {i}| {line}")

# Retrieve the plugin through the registry to prove the registration worked.
loaded = registry.get_loaded_object("minijs")
assert loaded is plugin, "registry round-trip failed"

ast = loaded.parse(snippet)
print()
print("AST      :", ast)
print("Symbols  :", loaded.extract_symbols(ast))
print("Imports  :", loaded.resolve_imports(ast))

Source:
  1| import { Widget } from 'widgets';
  2| const MAX_ITEMS = 10;
  3| function renderList(items) { return items.slice(0, MAX_ITEMS); }

AST      : {'language': 'minijs', 'nodes': [{'kind': 'const', 'name': 'MAX_ITEMS', 'lineno': 2}, {'kind': 'function', 'name': 'renderList', 'lineno': 3}], 'imports': ['widgets']}
Symbols  : [{'name': 'MAX_ITEMS', 'kind': 'const', 'line_start': 2}, {'name': 'renderList', 'kind': 'function', 'line_start': 3}]
Imports  : ['widgets']


## 4. Add the parsed nodes to a COGANT `ProgramGraph`

A language plugin stops at the AST boundary — turning AST nodes into `ProgramGraph` nodes is the pipeline's job. For a notebook demo we do it by hand so you can see exactly how the bridge works: each Mini-JS symbol becomes a `Node` with a matching `NodeKind`, and the `import` becomes an `IMPORTS` edge from a synthetic module node.

In [4]:
from cogant.schemas.core import Edge, EdgeKind, Node, NodeKind
from cogant.schemas.graph import GraphMetadata, ProgramGraph

graph = ProgramGraph(metadata=GraphMetadata(repo_uri="synthetic://minijs"))

# Module root node.
mod_id = "mod:snippet"
graph.add_node(Node(id=mod_id, kind=NodeKind.MODULE, name="snippet", qualified_name="snippet"))

KIND_MAP = {"function": NodeKind.FUNCTION, "const": NodeKind.VARIABLE}
DEFAULT_KIND = NodeKind.VARIABLE

symbol_ids: list[str] = []
for sym in loaded.extract_symbols(ast):
    nid = f"sym:{sym['name']}"
    graph.add_node(
        Node(
            id=nid,
            kind=KIND_MAP.get(sym["kind"], DEFAULT_KIND),
            name=sym["name"],
            qualified_name=f"snippet.{sym['name']}",
        )
    )
    graph.add_edge(
        Edge(
            id=f"e:contains:{sym['name']}", source_id=mod_id, target_id=nid, kind=EdgeKind.CONTAINS
        )
    )
    symbol_ids.append(nid)

# One IMPORTS edge per import target.
for i, target in enumerate(loaded.resolve_imports(ast)):
    target_id = f"ext:{target}"
    graph.add_node(Node(id=target_id, kind=NodeKind.MODULE, name=target, qualified_name=target))
    graph.add_edge(
        Edge(id=f"e:imports:{i}", source_id=mod_id, target_id=target_id, kind=EdgeKind.IMPORTS)
    )

print(f"Graph: {len(graph.nodes)} nodes, {len(graph.edges)} edges")
print("Nodes:")
for n in graph.nodes.values():
    print(f"  {n.id:<20} kind={n.kind.value:<12} name={n.name}")
print("Edges:")
for e in graph.edges.values():
    print(f"  {e.source_id} --[{e.kind.value}]--> {e.target_id}")

Graph: 4 nodes, 3 edges
Nodes:
  mod:snippet          kind=module       name=snippet
  sym:MAX_ITEMS        kind=variable     name=MAX_ITEMS
  sym:renderList       kind=function     name=renderList
  ext:widgets          kind=module       name=widgets
Edges:
  mod:snippet --[contains]--> sym:MAX_ITEMS
  mod:snippet --[contains]--> sym:renderList
  mod:snippet --[imports]--> ext:widgets


## 5. Sanity check

A successful run of this notebook should yield a graph with exactly:

- the module node,
- one FUNCTION node (`renderList`),
- one VARIABLE node (`MAX_ITEMS`),
- one external module node (`widgets`),
- two `CONTAINS` edges, and one `IMPORTS` edge.

If any of those counts drift, the regexes or the `KIND_MAP` need updating.

In [5]:
assert len(graph.nodes) == 4, f"expected 4 nodes, got {len(graph.nodes)}"
assert len(graph.edges) == 3, f"expected 3 edges, got {len(graph.edges)}"
kinds = sorted(n.kind.value for n in graph.nodes.values())
assert kinds.count("module") == 2, kinds
print("OK — plugin produced the expected graph shape.")

plugin.shutdown()

OK — plugin produced the expected graph shape.


## 6. Takeaways

- A minimal `LanguagePlugin` is ~50 lines of Python — no packaging, no entry points required for in-process use.
- The same `register_in_process` pattern works for `TracePlugin`, `RulePlugin`, and the other protocols under `cogant.plugins.base`.
- When you're ready to ship your plugin as a wheel, add the `[project.entry-points."cogant.plugins"]` stanza (see notebook 06) and `PluginRegistry.discover()` will pick it up automatically.